# UTILS

In [ ]:
%%writefile utils.py
import pandas as pd
import numpy as np
import os
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
import warnings
warnings.filterwarnings('ignore')

TRAIN_PATH = "/kaggle/input/playground-series-s5e12/train.csv"
TEST_PATH = "/kaggle/input/playground-series-s5e12/test.csv"
ORIG_PATH = "/kaggle/input/diabetes-health-indicators-dataset/diabetes_dataset.csv"
OUTPUT_DIR = "outputs"
N_SPLITS = 10
RANDOM_STATE = 42
CUTOFF_ID = 678260

os.makedirs(OUTPUT_DIR, exist_ok=True)

def load_data():
    train = pd.read_csv(TRAIN_PATH)
    test = pd.read_csv(TEST_PATH)
    orig = pd.read_csv(ORIG_PATH)
    
    BASE = [col for col in train.columns if col not in ['id', 'diagnosed_diabetes']]
    
    print("🔧 Adding Original Data Features...")
    
    # --- Original Data Mean/Count Encoding ---
    for col in BASE:
        if col in orig.columns:
            mean_map = orig.groupby(col)['diagnosed_diabetes'].mean()
            new_col_mean = f'orig_mean_{col}'
            
            train[new_col_mean] = train[col].map(mean_map)
            test[new_col_mean] = test[col].map(mean_map)
            
            global_mean = orig['diagnosed_diabetes'].mean()
            train[new_col_mean] = train[new_col_mean].fillna(global_mean)
            test[new_col_mean] = test[new_col_mean].fillna(global_mean)
            
            count_map = orig.groupby(col).size()
            new_col_count = f'orig_count_{col}'
            
            train[new_col_count] = train[col].map(count_map)
            test[new_col_count] = test[col].map(count_map)
            
            train[new_col_count] = train[new_col_count].fillna(0)
            test[new_col_count] = test[new_col_count].fillna(0)
    
    print(f"✅ Added {len(BASE) * 2} Original features")
    
    X = train.drop("diagnosed_diabetes", axis=1)
    y = train["diagnosed_diabetes"]
    
    return X, y, test 

def add_interaction_features(X, X_test):
    print("🔧 Generating Interaction Features...")
    
    n_train = len(X)
    df_all = pd.concat([X, X_test], axis=0).reset_index(drop=True)
    
    if 'systolic_bp' in df_all.columns and 'diastolic_bp' in df_all.columns:
        df_all['pulse_pressure'] = df_all['systolic_bp'] - df_all['diastolic_bp']
        df_all['map_bp'] = df_all['diastolic_bp'] + (df_all['pulse_pressure'] / 3)
    
    if 'age' in df_all.columns:
        df_all['age_bin'] = pd.qcut(df_all['age'], q=10, labels=False, duplicates='drop').astype(str)
    
    if 'bmi' in df_all.columns:
        df_all['bmi_bin'] = pd.qcut(df_all['bmi'], q=10, labels=False, duplicates='drop').astype(str)
        
    if 'glucose_fasting' in df_all.columns:
        df_all['glucose_bin'] = pd.qcut(df_all['glucose_fasting'], q=5, labels=False, duplicates='drop').astype(str)

    if 'age_bin' in df_all.columns and 'gender' in df_all.columns:
        df_all['gender_age_inter'] = df_all['gender'].astype(str) + '_' + df_all['age_bin']
        
    if 'ethnicity' in df_all.columns and 'bmi_bin' in df_all.columns:
        df_all['ethnicity_bmi_inter'] = df_all['ethnicity'].astype(str) + '_' + df_all['bmi_bin']
        
    if 'education_level' in df_all.columns and 'smoking_status' in df_all.columns:
        df_all['edu_smoke_inter'] = df_all['education_level'].astype(str) + '_' + df_all['smoking_status'].astype(str)
    
    if 'family_history_diabetes' in df_all.columns and 'age_bin' in df_all.columns:
        df_all['family_age_inter'] = df_all['family_history_diabetes'].astype(str) + '_' + df_all['age_bin']

    X_new = df_all.iloc[:n_train].copy()
    X_test_new = df_all.iloc[n_train:].copy()
    
    X_new.index = X.index
    X_test_new.index = X_test.index
    
    print(f"✅ Interaction Features added. Total cols: {X_new.shape[1]}")
    return X_new, X_test_new

def encode_categorical(X, X_test):
    cat_cols = X.select_dtypes(include=["object", "category"]).columns.tolist()
    
    n_train = len(X)
    df_all = pd.concat([X, X_test], axis=0)
    
    for col in cat_cols:
        df_all[col] = df_all[col].astype("category").cat.codes
        
    X_enc = df_all.iloc[:n_train]
    X_test_enc = df_all.iloc[n_train:]
    
    return X_enc, X_test_enc

def get_folds(X, y):
    skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)
    return list(skf.split(X, y))

def save_predictions(oof, pred, test_ids, name):
    pd.DataFrame({'oof': oof}).to_csv(f"{OUTPUT_DIR}/oof_{name}.csv", index=False)
    pd.DataFrame({'id': test_ids, 'diagnosed_diabetes': pred}).to_csv(f"{OUTPUT_DIR}/submission_{name}.csv", index=False)
    
    train = pd.read_csv(TRAIN_PATH)
    y_true = train['diagnosed_diabetes']
    cutoff_mask = train['id'].values >= CUTOFF_ID
    
    full_auc = roc_auc_score(y_true, oof)
    cutoff_auc = roc_auc_score(y_true[cutoff_mask], oof[cutoff_mask])
    
    print(f"\n{'='*50}")
    print(f"✅ {name.upper()} saved")
    print(f"📊 Full OOF AUC:   {full_auc:.5f}")
    print(f"⭐ Cutoff AUC:     {cutoff_auc:.5f} (id >= {CUTOFF_ID}, {cutoff_mask.sum()} samples)")
    print(f"{'='*50}")

# MODEL

In [ ]:
%%writefile train_lightgbm.py
import numpy as np
import lightgbm as lgb
from utils import load_data, encode_categorical, add_interaction_features, get_folds, save_predictions, RANDOM_STATE
from sklearn.metrics import roc_auc_score

print("🚀 Training LightGBM with Interactions...")

X, y, test = load_data()

X, test = add_interaction_features(X, test)
X, X_test = encode_categorical(X.copy(), test.copy())

params = {
    'objective': 'binary',
    'metric': 'auc',
    'learning_rate': 0.03,
    'num_leaves': 63,
    'feature_fraction': 0.8,
    'bagging_fraction': 0.8,
    'bagging_freq': 1,
    'n_estimators': 10000, 
    'device': 'cpu',
    'verbose': -1,
    'random_state': RANDOM_STATE
}

oof = np.zeros(len(X))
pred = np.zeros(len(X_test))

for fold, (trn_idx, val_idx) in enumerate(get_folds(X, y)):
    print(f"\n⚡ Fold {fold+1}/10")
    
    X_tr, X_val = X.iloc[trn_idx], X.iloc[val_idx]
    y_tr, y_val = y.iloc[trn_idx], y.iloc[val_idx]
    
    model = lgb.LGBMClassifier(**params)
    model.fit(
        X_tr, y_tr,
        eval_set=[(X_val, y_val)],
        callbacks=[
            lgb.early_stopping(stopping_rounds=500, verbose=True),
            lgb.log_evaluation(period=1000)
        ]
    )
    
    oof[val_idx] = model.predict_proba(X_val)[:, 1]
    pred += model.predict_proba(X_test)[:, 1] / 10
    
    fold_auc = roc_auc_score(y_val, oof[val_idx])
    print(f"✅ Fold {fold+1} stopped at {model.best_iteration_} trees, AUC: {fold_auc:.5f}")

save_predictions(oof, pred, test['id'], 'lgb')

In [ ]:
%%writefile train_xgboost.py
import numpy as np
import xgboost as xgb
from utils import load_data, encode_categorical, get_folds, save_predictions, RANDOM_STATE
from sklearn.metrics import roc_auc_score

print("🚀 Training XGBoost...")

X, y, test = load_data()
X, X_test = encode_categorical(X.copy(), test.copy())

params = {
    'n_estimators': 10000, 
    'max_depth': 5,
    'learning_rate': 0.03,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'eval_metric': 'auc',
    'random_state': RANDOM_STATE,
    'tree_method': 'hist',
    'device': 'cpu'
}

oof = np.zeros(len(X))
pred = np.zeros(len(X_test))

for fold, (trn_idx, val_idx) in enumerate(get_folds(X, y)):
    print(f"\n⚡ Fold {fold+1}/10")
    
    X_tr, X_val = X.iloc[trn_idx], X.iloc[val_idx]
    y_tr, y_val = y.iloc[trn_idx], y.iloc[val_idx]
    
    model = xgb.XGBClassifier(**params)
    model.fit(
        X_tr, y_tr,
        eval_set=[(X_val, y_val)],
        early_stopping_rounds=500,
        verbose=1000
    )
    
    oof[val_idx] = model.predict_proba(X_val)[:, 1]
    pred += model.predict_proba(X_test)[:, 1] / 10
    
    fold_auc = roc_auc_score(y_val, oof[val_idx])
    print(f"✅ Fold {fold+1} stopped at {model.best_iteration} trees, AUC: {fold_auc:.5f}")

save_predictions(oof, pred, test['id'], 'xgb')

In [ ]:
%%writefile train_lgb_te.py
import numpy as np
import pandas as pd
import lightgbm as lgb
from category_encoders import TargetEncoder 
from sklearn.metrics import roc_auc_score
from utils import load_data, get_folds, save_predictions, RANDOM_STATE, TRAIN_PATH, ORIG_PATH, CUTOFF_ID
import warnings

warnings.filterwarnings('ignore')

# 配置
MODEL_NAME = 'lgb_te'
TE_SMOOTHING = 50.0   
MIN_SAMPLES_LEAF = 10 

print(f"🚀 Training LightGBM (Tail-Based Target Encoding)...")

X, y, test = load_data()
orig = pd.read_csv(ORIG_PATH) 

def preprocess_s5e11(df):
    cont_cols = ['bmi', 'physical_activity_minutes_per_week', 'sleep_hours_per_day', 
                 'cholesterol_total', 'ldl_cholesterol', 'hdl_cholesterol', 
                 'systolic_bp', 'diastolic_bp', 'triglycerides']
    
    cont_cols = [c for c in cont_cols if c in df.columns]
    
    # 简单的分箱和字符处理
    for col in cont_cols:
        try:
            df[f'{col}_qcut'] = pd.qcut(df[col], q=20, labels=False, duplicates='drop').astype(str)
        except:
            pass 
        # 保留原有数值的一半作为一种模糊化
        df[f'{col}_round_div2'] = (df[col].round() // 2).astype(int).astype(str)

    return df

n_train = len(X)
full_df = pd.concat([X, test], axis=0).reset_index(drop=True)
full_df = preprocess_s5e11(full_df)

X = full_df.iloc[:n_train]
test = full_df.iloc[n_train:]

te_cols = [c for c in X.columns if X[c].dtype == 'object' or X[c].dtype.name == 'category']
orig_cat_cols = ['gender', 'ethnicity', 'education_level', 'smoking_status', 'family_history_diabetes']
te_cols = list(set(te_cols + [c for c in orig_cat_cols if c in X.columns]))

print(f"🔧 Features generated. Columns to Tail-Target Encode: {len(te_cols)}")

train_ids = pd.read_csv(TRAIN_PATH)['id'].values

common_cols = [c for c in X.columns if c in orig.columns]
X_orig = orig[common_cols].copy()
y_orig = orig['diagnosed_diabetes']
X_orig = preprocess_s5e11(X_orig) 
X_orig_te = X_orig[te_cols]

# 3. 模型参数 (保持重正则化)
params = {
    'objective': 'binary',
    'metric': 'auc',
    'n_estimators': 10000,
    'learning_rate': 0.01,
    'max_depth': 4,
    'subsample': 0.6,        
    'colsample_bytree': 0.4,  
    'lambda_l1': 10.0,
    'lambda_l2': 15.0,
    'min_child_samples': 50,
    'random_state': RANDOM_STATE,
    'n_jobs': 4,
    'device': 'cpu',
    'verbose': -1
}

oof = np.zeros(len(X))
pred = np.zeros(len(test))

folds = get_folds(X, y)

for fold, (trn_idx, val_idx) in enumerate(folds):
    print(f"\n⚡ Fold {fold+1}/10")
    
    X_tr = X.iloc[trn_idx].copy()
    y_tr = y.iloc[trn_idx]
    X_val = X.iloc[val_idx].copy()
    y_val = y.iloc[val_idx]
    X_test_fold = test.copy()
    
    tr_ids = train_ids[trn_idx]
    
    
    tail_mask = tr_ids >= CUTOFF_ID
    
    X_tr_tail = X_tr[tail_mask][te_cols]
    y_tr_tail = y_tr[tail_mask]
    
    # 拼接 Tail 和 Orig 用于计算统计量
    X_fit_te = pd.concat([X_tr_tail, X_orig_te], axis=0)
    y_fit_te = pd.concat([y_tr_tail, y_orig], axis=0)
    
    print(f"   TE Fitting Size: {len(X_fit_te)} (Tail: {len(X_tr_tail)} + Orig: {len(X_orig_te)})")
    
    if len(te_cols) > 0:
        encoder = TargetEncoder(
            cols=te_cols, 
            smoothing=TE_SMOOTHING, 
            min_samples_leaf=MIN_SAMPLES_LEAF
        )
        
        encoder.fit(X_fit_te, y_fit_te)
        
        X_tr[te_cols] = encoder.transform(X_tr[te_cols])
        X_val[te_cols] = encoder.transform(X_val[te_cols])
        X_test_fold[te_cols] = encoder.transform(X_test_fold[te_cols])
    
    for col in te_cols:
        X_tr[col] = X_tr[col].astype(float)
        X_val[col] = X_val[col].astype(float)
        X_test_fold[col] = X_test_fold[col].astype(float)

    model = lgb.LGBMClassifier(**params)
    model.fit(
        X_tr, y_tr,
        eval_set=[(X_val, y_val)],
        callbacks=[
            lgb.early_stopping(stopping_rounds=500, verbose=True),
            lgb.log_evaluation(period=2000)
        ]
    )
    
    oof[val_idx] = model.predict_proba(X_val)[:, 1]
    pred += model.predict_proba(X_test_fold)[:, 1] / 10
    
    fold_auc = roc_auc_score(y_val, oof[val_idx])
    
    val_ids = train_ids[val_idx]
    val_tail_mask = val_ids >= CUTOFF_ID
    if val_tail_mask.sum() > 0:
        tail_auc = roc_auc_score(y_val[val_tail_mask], oof[val_idx][val_tail_mask])
        print(f"✅ Fold {fold+1} Full AUC: {fold_auc:.5f} | Tail AUC: {tail_auc:.5f}")
    else:
        print(f"✅ Fold {fold+1} Full AUC: {fold_auc:.5f}")

save_predictions(oof, pred, pd.read_csv("/kaggle/input/playground-series-s5e12/test.csv")['id'], MODEL_NAME)

In [ ]:
%%writefile train_lgb_safe.py
import numpy as np
import pandas as pd
import lightgbm as lgb
from sklearn.metrics import roc_auc_score
from utils import load_data, encode_categorical, add_interaction_features, get_folds, save_predictions, RANDOM_STATE, CUTOFF_ID

MODEL_NAME = 'lgb_safe'

TOXIC_FEATURES = [
    'physical_activity_minutes_per_week', # Shift 0.034 (Too high)
    'sleep_hours_per_day',                # Signal lost (0.004 -> -0.001)
    'heart_rate'                          # Signal decayed by 50% (0.022 -> 0.011)
]

print(f"🚀 Training LightGBM on SAFE Features Only (Dropping {len(TOXIC_FEATURES)} toxic features)...")

X, y, test = load_data()

X, test = add_interaction_features(X, test)
X, X_test = encode_categorical(X.copy(), test.copy())

cols_to_drop = []
all_cols = X.columns.tolist()

for col in all_cols:
    if col in TOXIC_FEATURES:
        cols_to_drop.append(col)
        continue
    
    for toxic in TOXIC_FEATURES:
        if toxic in col: 
             cols_to_drop.append(col)
             break

print(f"⚠️ Dropping {len(cols_to_drop)} features containing toxic signals.")
print(f"Examples: {cols_to_drop[:5]}")

X = X.drop(columns=cols_to_drop)
X_test = X_test.drop(columns=cols_to_drop)

print(f"✅ Final Feature Count: {X.shape[1]}")

params = {
    'objective': 'binary',
    'metric': 'auc',
    'learning_rate': 0.03,
    'num_leaves': 63, 
    'feature_fraction': 0.8,
    'bagging_fraction': 0.8,
    'bagging_freq': 1,
    'n_estimators': 10000,
    'device': 'cpu',
    'verbose': -1,
    'random_state': RANDOM_STATE
}

# 5. 训练
oof = np.zeros(len(X))
pred = np.zeros(len(X_test))
train_ids = pd.read_csv("/kaggle/input/playground-series-s5e12/train.csv")['id'].values

for fold, (trn_idx, val_idx) in enumerate(get_folds(X, y)):
    print(f"\n⚡ Fold {fold+1}/10")
    X_tr, X_val = X.iloc[trn_idx], X.iloc[val_idx]
    y_tr, y_val = y.iloc[trn_idx], y.iloc[val_idx]

    model = lgb.LGBMClassifier(**params)
    model.fit(
        X_tr, y_tr,
        eval_set=[(X_val, y_val)],
        callbacks=[
            lgb.early_stopping(stopping_rounds=500, verbose=True),
            lgb.log_evaluation(period=1000)
        ]
    )
    
    oof[val_idx] = model.predict_proba(X_val)[:, 1]
    pred += model.predict_proba(X_test)[:, 1] / 10
    
    fold_auc = roc_auc_score(y_val, oof[val_idx])
    
    # 计算当前 Fold 的 Tail AUC 监控效果
    val_ids = train_ids[val_idx]
    val_tail_mask = val_ids >= CUTOFF_ID
    if val_tail_mask.sum() > 0:
        tail_auc = roc_auc_score(y_val[val_tail_mask], oof[val_idx][val_tail_mask])
        print(f"✅ Fold {fold+1} Full AUC: {fold_auc:.5f} | Tail AUC: {tail_auc:.5f}")
    else:
        print(f"✅ Fold {fold+1} AUC: {fold_auc:.5f}")

# 6. 保存与评估
save_predictions(oof, pred, pd.read_csv("/kaggle/input/playground-series-s5e12/test.csv")['id'], MODEL_NAME)

In [ ]:
%%writefile train_lgb_pseudo.py
import os
import numpy as np
import pandas as pd
import lightgbm as lgb
from sklearn.metrics import roc_auc_score
from scipy.stats import rankdata
from utils import load_data, encode_categorical, add_interaction_features, get_folds, save_predictions, RANDOM_STATE, TRAIN_PATH, CUTOFF_ID

MODEL_NAME = 'lgb_pseudo'
PSEUDO_THR_HIGH = 0.95 
PSEUDO_THR_LOW = 0.03  
# 注意后面的编号是LB分数，比如LB分数为0.70717就是submission_0_70717.csv文件
BEST_SUB_PATH = "/kaggle/input/ps-s5e12-oofs-subs/submission_0_70800.csv" 

print(f"🚀 Training LightGBM with Pseudo-Labelings (With Distribution Fix)...")

X, y, test = load_data()
X, test = add_interaction_features(X, test)
X, X_test = encode_categorical(X.copy(), test.copy())

train_raw = pd.read_csv(TRAIN_PATH)
train_ids = train_raw['id'].values
tail_mask = train_ids >= CUTOFF_ID
head_mask = ~tail_mask

# Split Data
X_tail = X[tail_mask].copy()
y_tail = y[tail_mask].copy()
X_head = X[head_mask].copy()
y_head = y[head_mask].copy()

print(f" Tail samples (Valid/Core Train): {len(X_tail)}")

if os.path.exists(BEST_SUB_PATH):
    sub_df = pd.read_csv(BEST_SUB_PATH)
    test_preds = sub_df['diagnosed_diabetes'].values

    pseudo_mask_pos = test_preds >= PSEUDO_THR_HIGH
    pseudo_mask_neg = test_preds <= PSEUDO_THR_LOW

    X_pseudo_pos = X_test[pseudo_mask_pos].copy()
    y_pseudo_pos = pd.Series(1, index=X_pseudo_pos.index)

    X_pseudo_neg = X_test[pseudo_mask_neg].copy()
    y_pseudo_neg = pd.Series(0, index=X_pseudo_neg.index)

    X_pseudo = pd.concat([X_pseudo_pos, X_pseudo_neg])
    y_pseudo = pd.concat([y_pseudo_pos, y_pseudo_neg])

    print(f"📌 Pseudo-Labels Generated: {len(X_pseudo)} samples (Pos: {len(X_pseudo_pos)}, Neg: {len(X_pseudo_neg)})")
else:
    print("⚠️ Pseudo source file not found, using empty pseudo set.")
    X_pseudo = pd.DataFrame(columns=X_tail.columns)
    y_pseudo = pd.Series(dtype=float)

params = {
    'objective': 'binary',
    'metric': 'auc',
    'learning_rate': 0.02,
    'num_leaves': 31,          
    'feature_fraction': 0.7,
    'bagging_fraction': 0.7,
    'bagging_freq': 1,
    'n_estimators': 5000,
    'lambda_l1': 1.0,
    'lambda_l2': 1.0,
    'min_child_samples': 50,
    'device': 'cpu',
    'verbose': -1,
    'random_state': RANDOM_STATE
}

oof = np.zeros(len(X)) 
pred_test = np.zeros(len(X_test))
oof_head_accum = np.zeros(len(X_head))

from sklearn.model_selection import StratifiedKFold
skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=RANDOM_STATE)

X_tail_reset = X_tail.reset_index(drop=True)
y_tail_reset = y_tail.reset_index(drop=True)

oof_tail_only = np.zeros(len(X_tail))

for fold, (trn_idx_tail, val_idx_tail) in enumerate(skf.split(X_tail_reset, y_tail_reset)):
    print(f"\n⚡ Fold {fold+1}/10")
    
    X_tr_fold = X_tail_reset.iloc[trn_idx_tail]
    y_tr_fold = y_tail_reset.iloc[trn_idx_tail]
    
    if len(X_pseudo) > 0:
        X_train_iter = pd.concat([X_tr_fold, X_pseudo], axis=0)
        y_train_iter = pd.concat([y_tr_fold, y_pseudo], axis=0)
    else:
        X_train_iter, y_train_iter = X_tr_fold, y_tr_fold
    
    X_val_fold = X_tail_reset.iloc[val_idx_tail]
    y_val_fold = y_tail_reset.iloc[val_idx_tail]
    
    model = lgb.LGBMClassifier(**params)
    model.fit(
        X_train_iter, y_train_iter,
        eval_set=[(X_val_fold, y_val_fold)],
        callbacks=[
            lgb.early_stopping(stopping_rounds=200, verbose=True),
            lgb.log_evaluation(period=0) 
        ]
    )

    val_preds = model.predict_proba(X_val_fold)[:, 1]
    
    oof_tail_only[val_idx_tail] = val_preds
    
    original_val_idx = X_tail.iloc[val_idx_tail].index
    oof[original_val_idx] = val_preds
    
    oof_head_accum += model.predict_proba(X_head)[:, 1] / 10
    
    pred_test += model.predict_proba(X_test)[:, 1] / 10
    
    fold_auc = roc_auc_score(y_val_fold, val_preds)
    print(f"✅ Fold {fold+1} Tail AUC: {fold_auc:.5f}")

oof[head_mask] = oof_head_accum

valid_auc = roc_auc_score(y_tail, oof[tail_mask])
print(f"\n📊 Raw Pseudo-Labeling Model Tail AUC: {valid_auc:.5f}")

print("\n🔧 Applying Quantile Mapping to fix KS & Scale...")

def quantile_map(source_preds, target_dist):
    rank = rankdata(source_preds, method='ordinal') - 1
    pct = rank / (len(source_preds) - 1)
    
    target_sorted = np.sort(target_dist)
    # 线性插值
    mapped = np.interp(pct, np.linspace(0, 1, len(target_sorted)), target_sorted)
    return mapped

target_dist = oof[tail_mask]

oof_mapped = quantile_map(oof, target_dist)

pred_test_mapped = quantile_map(pred_test, target_dist)

oof_mapped[tail_mask] = oof[tail_mask] 

# 验证
final_auc = roc_auc_score(y, oof_mapped)
final_cutoff_auc = roc_auc_score(y[tail_mask], oof_mapped[tail_mask])

print(f"📊 Mapped Full OOF AUC: {final_auc:.5f}")
print(f"⭐ Mapped Cutoff AUC:   {final_cutoff_auc:.5f}")

# 保存
save_predictions(oof_mapped, pred_test_mapped, test['id'], MODEL_NAME)

In [ ]:
%%writefile train_nn_dae.py
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
import os
import gc
from utils import load_data, save_predictions, RANDOM_STATE, CUTOFF_ID

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
BATCH_SIZE = 1024
LR_DAE = 2e-3      
LR_SUP = 1e-3      
LR_FT = 1e-4       
EPOCHS_DAE = 25    
EPOCHS_SUP = 15    
EPOCHS_FT = 15     
HIDDEN_DIM = 512
NOISE_LEVEL = 0.15 

print(f"🚀 Training NN (Entity Embeddings + DAE w/ Test) on {DEVICE}...")

X, y, test = load_data()
train_ids = pd.read_csv("/kaggle/input/playground-series-s5e12/train.csv")['id'].values

cat_cols = [c for c in X.columns if X[c].dtype == 'object' or X[c].dtype.name == 'category']
num_cols = [c for c in X.columns if c not in cat_cols]

print(f"   Cat cols: {len(cat_cols)}, Num cols: {len(num_cols)}")

cat_dims = [] 

for col in cat_cols:
    X[col] = X[col].fillna("MISSING").astype(str)
    test[col] = test[col].fillna("MISSING").astype(str)
    
    le = LabelEncoder()
    full = pd.concat([X[col], test[col]])
    le.fit(full)
    
    X[col] = le.transform(X[col])
    test[col] = le.transform(test[col])
    
    cat_dims.append(len(le.classes_))

scaler = StandardScaler()
X[num_cols] = scaler.fit_transform(X[num_cols])
test[num_cols] = scaler.transform(test[num_cols])

X_num = X[num_cols].values.astype(np.float32)
X_cat = X[cat_cols].values.astype(np.int64) 
y_np = y.values.astype(np.float32)

test_num = test[num_cols].values.astype(np.float32)
test_cat = test[cat_cols].values.astype(np.int64)

num_features = X_num.shape[1]

class TabularDataset(Dataset):
    def __init__(self, x_num, x_cat, y=None, is_dae=False, noise_level=0.15):
        self.x_num = torch.FloatTensor(x_num)
        self.x_cat = torch.LongTensor(x_cat)
        self.y = torch.FloatTensor(y).unsqueeze(1) if y is not None else None
        self.is_dae = is_dae
        self.noise_level = noise_level
        
    def __len__(self):
        return len(self.x_num)
    
    def __getitem__(self, idx):
        x_n = self.x_num[idx]
        x_c = self.x_cat[idx]
        
        if self.is_dae:
            noise = torch.randn_like(x_n) * self.noise_level
            x_n_noisy = x_n + noise
            return x_n_noisy, x_c, x_n 
        
        if self.y is not None:
            return x_n, x_c, self.y[idx]
        return x_n, x_c

# --- Model ---
class DAE_Embedding_Net(nn.Module):
    def __init__(self, num_dim, cat_dims, hidden_dim):
        super().__init__()
        
        self.embeddings = nn.ModuleList([
            nn.Embedding(num_embeddings=d, embedding_dim=min(50, (d+1)//2))
            for d in cat_dims
        ])
        
        total_emb_dim = sum([e.embedding_dim for e in self.embeddings])
        input_dim = num_dim + total_emb_dim
        
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.SiLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.SiLU(),
        )
        
        self.decoder_num = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.SiLU(),
            nn.Linear(hidden_dim, num_dim)
        )
        
        self.head = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.BatchNorm1d(hidden_dim // 2),
            nn.SiLU(),
            nn.Dropout(0.3),
            nn.Linear(hidden_dim // 2, 1)
        )
        
    def forward_features(self, x_num, x_cat):
        emb_list = []
        for i, emb_layer in enumerate(self.embeddings):
            emb_list.append(emb_layer(x_cat[:, i]))
        x_emb = torch.cat(emb_list, dim=1)
        x = torch.cat([x_num, x_emb], dim=1)
        feat = self.encoder(x)
        return feat

    def forward_dae(self, x_num, x_cat):
        feat = self.forward_features(x_num, x_cat)
        recon_num = self.decoder_num(feat)
        return recon_num
    
    def forward_clf(self, x_num, x_cat):
        feat = self.forward_features(x_num, x_cat)
        return torch.sigmoid(self.head(feat))

oof = np.zeros(len(X))
pred_test = np.zeros(len(test))

X_all_num = np.vstack([X_num, test_num])
X_all_cat = np.vstack([X_cat, test_cat])

skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=RANDOM_STATE)

for fold, (trn_idx, val_idx) in enumerate(skf.split(X, y)):
    print(f"\n⚡ Fold {fold+1}/10")
    
    X_num_tr, X_cat_tr, y_tr = X_num[trn_idx], X_cat[trn_idx], y_np[trn_idx]
    X_num_val, X_cat_val, y_val = X_num[val_idx], X_cat[val_idx], y_np[val_idx]
    
    tr_ids_fold = train_ids[trn_idx]
    val_ids_fold = train_ids[val_idx]
    
    tr_tail_mask = tr_ids_fold >= CUTOFF_ID
    val_tail_mask = val_ids_fold >= CUTOFF_ID
    has_tail_in_val = val_tail_mask.sum() > 10
    
    X_num_tail = X_num_tr[tr_tail_mask]
    X_cat_tail = X_cat_tr[tr_tail_mask]
    y_tail = y_tr[tr_tail_mask]
    
    model = DAE_Embedding_Net(num_features, cat_dims, HIDDEN_DIM).to(DEVICE)
    
    print("   Phase 0: DAE Pretraining (Reconstruct Num)...")
    dae_ds = TabularDataset(X_all_num, X_all_cat, is_dae=True, noise_level=NOISE_LEVEL)
    dae_loader = DataLoader(dae_ds, batch_size=BATCH_SIZE, shuffle=True)
    
    optimizer_dae = optim.AdamW(model.parameters(), lr=LR_DAE)
    criterion_dae = nn.MSELoss()
    
    model.train()
    for epoch in range(EPOCHS_DAE):
        for x_n_noisy, x_c, x_n_true in dae_loader:
            x_n_noisy, x_c, x_n_true = x_n_noisy.to(DEVICE), x_c.to(DEVICE), x_n_true.to(DEVICE)
            optimizer_dae.zero_grad()
            recon = model.forward_dae(x_n_noisy, x_c)
            loss = criterion_dae(recon, x_n_true)
            loss.backward()
            optimizer_dae.step()
            
    print("   Phase 1: Supervised Full Training...")
    sup_loader = DataLoader(TabularDataset(X_num_tr, X_cat_tr, y_tr), batch_size=BATCH_SIZE, shuffle=True)
    val_loader = DataLoader(TabularDataset(X_num_val, X_cat_val, y_val), batch_size=BATCH_SIZE*2, shuffle=False)
    
    optimizer_sup = optim.AdamW(model.parameters(), lr=LR_SUP)
    criterion_sup = nn.BCELoss()
    
    best_auc = 0
    best_state = None
    
    for epoch in range(EPOCHS_SUP):
        model.train()
        for x_n, x_c, label in sup_loader:
            x_n, x_c, label = x_n.to(DEVICE), x_c.to(DEVICE), label.to(DEVICE)
            optimizer_sup.zero_grad()
            pred = model.forward_clf(x_n, x_c)
            loss = criterion_sup(pred, label)
            loss.backward()
            optimizer_sup.step()
            
        model.eval()
        preds = []
        with torch.no_grad():
            for x_n, x_c, _ in val_loader:
                preds.append(model.forward_clf(x_n.to(DEVICE), x_c.to(DEVICE)).cpu().numpy())
        epoch_auc = roc_auc_score(y_val, np.vstack(preds))
        
        if epoch_auc > best_auc:
            best_auc = epoch_auc
            best_state = model.state_dict().copy()
            
    print(f"      Stage 1 Best Global AUC: {best_auc:.5f}")
    
    print(f"   Phase 2: Finetuning on Tail ({len(y_tail)} samples)...")
    model.load_state_dict(best_state)
    
    ft_loader = DataLoader(TabularDataset(X_num_tail, X_cat_tail, y_tail), batch_size=BATCH_SIZE//2, shuffle=True)
    optimizer_ft = optim.AdamW(model.parameters(), lr=LR_FT)
    
    model.eval()
    preds = []
    with torch.no_grad():
        for x_n, x_c, _ in val_loader:
            preds.append(model.forward_clf(x_n.to(DEVICE), x_c.to(DEVICE)).cpu().numpy())
    full_preds = np.vstack(preds).ravel()
    
    baseline_tail_auc = 0
    if has_tail_in_val:
        baseline_tail_auc = roc_auc_score(y_val[val_tail_mask], full_preds[val_tail_mask])
        print(f"      Baseline Tail AUC: {baseline_tail_auc:.5f}")
    
    final_state = best_state
    best_tail_auc = baseline_tail_auc
    
    for epoch in range(EPOCHS_FT):
        model.train()
        for x_n, x_c, label in ft_loader:
            x_n, x_c, label = x_n.to(DEVICE), x_c.to(DEVICE), label.to(DEVICE)
            optimizer_ft.zero_grad()
            pred = model.forward_clf(x_n, x_c)
            loss = criterion_sup(pred, label)
            loss.backward()
            optimizer_ft.step()
            
        if has_tail_in_val:
            model.eval()
            preds = []
            with torch.no_grad():
                for x_n, x_c, _ in val_loader:
                    preds.append(model.forward_clf(x_n.to(DEVICE), x_c.to(DEVICE)).cpu().numpy())
            full_preds = np.vstack(preds).ravel()
            
            curr_auc = roc_auc_score(y_val[val_tail_mask], full_preds[val_tail_mask])
            
            if curr_auc > best_tail_auc:
                best_tail_auc = curr_auc
                final_state = model.state_dict().copy()
                
    if has_tail_in_val:
        print(f"      Stage 2 Best Tail AUC: {best_tail_auc:.5f} (Delta: {best_tail_auc - baseline_tail_auc:+.5f})")

    model.load_state_dict(final_state)
    model.eval()
    
    preds = []
    with torch.no_grad():
        for x_n, x_c, _ in val_loader:
            preds.append(model.forward_clf(x_n.to(DEVICE), x_c.to(DEVICE)).cpu().numpy())
    oof[val_idx] = np.vstack(preds).ravel()
    
    test_loader = DataLoader(TabularDataset(test_num, test_cat, None), batch_size=BATCH_SIZE*2, shuffle=False)
    preds = []
    with torch.no_grad():
        for x_n, x_c in test_loader:
            preds.append(model.forward_clf(x_n.to(DEVICE), x_c.to(DEVICE)).cpu().numpy())
    pred_test += np.vstack(preds).ravel() / 10

save_predictions(oof, pred_test, pd.read_csv("/kaggle/input/playground-series-s5e12/test.csv")['id'], 'nn_dae')

In [ ]:
%%writefile train_xgb_weighted.py
import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
import warnings
from utils import load_data, save_predictions, TRAIN_PATH, ORIG_PATH, CUTOFF_ID, RANDOM_STATE

warnings.filterwarnings('ignore')

MODEL_NAME = 'xgb_weighted'
print(f"🚀 Training XGBoost (Strict Reproduction of Reference)...")


xgb_params = {
    'tree_method': 'hist',
    'objective': 'binary:logistic',
    'random_state': 42,
    'enable_categorical': True,
    'verbosity': 0,
    'eval_metric': 'auc',
    'booster': 'gbtree',
    'n_jobs': 4,
    'learning_rate': 0.01,
    'device': 'cpu',  # Kaggle GPU限额时改用CPU，如果有GPU可改为cuda
    'lambda': 1.134330929497114,
    'alpha': 6.780537184218281,
    'colsample_bytree': 0.1007615968427798,
    'subsample': 0.7215917619002097,
    'max_depth': 4,
    'min_child_weight': 5
}

# ======================================================================================
# 2. 数据加载与预处理 (复刻 Preprocessing 类逻辑)
# ======================================================================================
def preprocessing_repro():
    # 读取原始文件，不使用 utils.load_data 的预处理，以免逻辑冲突
    train = pd.read_csv(TRAIN_PATH)
    test = pd.read_csv("/kaggle/input/playground-series-s5e12/test.csv")
    orig = pd.read_csv(ORIG_PATH).dropna()
    
    # 对齐列名 (Orig 中通常是 Diabetes_binary 或 similar，这里假设列名已对齐或做映射)
    # 标准数据集列名为 'Diabetes_binary'，竞赛数据为 'diagnosed_diabetes'
    if 'Diabetes_binary' in orig.columns:
        orig = orig.rename(columns={'Diabetes_binary': 'diagnosed_diabetes'})
    
    # 确保列一致
    common_cols = [c for c in train.columns if c in orig.columns]
    orig = orig[common_cols]
    
    target = 'diagnosed_diabetes'
    
    # 记录原始数据长度以便后续切分
    n_train = len(train)
    n_test = len(test)
    n_orig = len(orig)
    
    # 拼接 Train + Orig (参考代码的操作: train = pd.concat([train, orig...]))
    # 注意：参考代码是在 split 之前就把 orig 拼进去了
    train_full = pd.concat([train, orig], axis=0, ignore_index=True)
    
    # 再次拼接 Train_Full + Test 用于特征工程 (参考 feature_engineering 函数)
    # combine = pd.concat([self.X, self.test])
    full_df = pd.concat([train_full, test], axis=0, ignore_index=True)
    
    # 定义特征列
    cat_features = full_df.select_dtypes(include=['object', 'bool']).columns.tolist()
    # 手动添加参考代码中指定的分类特征
    extra_cats = ['family_history_diabetes', 'hypertension_history', 'cardiovascular_history']
    for c in extra_cats:
        if c not in cat_features and c in full_df.columns:
            cat_features.append(c)
            
    num_features = [c for c in full_df.columns if c not in cat_features and c != target and c != 'id']
    
    print("🔧 Applying Reference Feature Engineering...")
    
    # --- 逻辑 1: 基于 Orig 的统计特征 (Mean & Count) ---
    # 对应参考代码: for c in self.num_features + self.cat_features: ...
    global_mean = orig[target].mean()
    
    for c in num_features + cat_features:
        # Mean Encoding from Orig
        mean_map = orig.groupby(c)[target].mean()
        col_mean = f'{c}_org_mean'
        full_df[col_mean] = full_df[c].map(mean_map).fillna(global_mean)
        
        # Count Encoding from Orig
        count_map = orig.groupby(c)[target].count()
        col_count = f'{c}_org_count'
        full_df[col_count] = full_df[c].map(count_map).fillna(0)
        
    # --- 逻辑 2: 频率编码 (Frequency Encoding) ---
    # 对应参考代码: for c in self.cat_features: freqs = df[c].value_counts(normalize=True)...
    for c in cat_features:
        freqs = full_df[c].value_counts(normalize=True)
        full_df[f"{c}_fe"] = full_df[c].map(freqs)
        
    # --- 逻辑 3: 类型转换 ---
    # 对应参考代码: df[self.cat_features] = df[self.cat_features].astype('category')
    for c in cat_features:
        full_df[c] = full_df[c].astype('category')
        
    # 拆分回 Train_Full (含 Orig) 和 Test
    X_train_full = full_df.iloc[:n_train + n_orig].copy()
    X_test = full_df.iloc[n_train + n_orig:].copy()
    
    return X_train_full, X_test, n_train, n_orig, cat_features

X_full, X_test, n_train, n_orig, cat_cols = preprocessing_repro()
y_full = X_full['diagnosed_diabetes']
X_full = X_full.drop(columns=['diagnosed_diabetes', 'id'])
X_test = X_test.drop(columns=['diagnosed_diabetes', 'id']) 

print("⚖️ Constructing Weights...")

# 1. Train 部分的权重 (前 n_train 行)
# 根据 ID 切分 Head 和 Tail
train_raw = pd.read_csv(TRAIN_PATH)
tail_mask = train_raw['id'] >= CUTOFF_ID

w_train = np.ones(n_train)
w_train[tail_mask] = 16  # Tail 权重 16

# 2. Orig 部分的权重 (后 n_orig 行)
w_orig = np.full(n_orig, 8) # Orig 权重 8

# 3. 拼接
weights = np.concatenate([w_train, w_orig])

print(f" Weights Shape: {weights.shape}, X_full Shape: {X_full.shape}")
print(f" Weight distribution: 1s={np.sum(weights==1)}, 16s={np.sum(weights==16)}, 8s={np.sum(weights==8)}")

# ======================================================================================
# 4. 训练与验证 (StratifiedKFold on Combined Data)
# ======================================================================================
# 关键点：我们需要对 (Train + Orig) 进行 CV，但最后保存 OOF 时只取 Train 部分 (前 n_train 行)
skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

oof_combined = np.zeros(len(X_full))
pred_test = np.zeros(len(X_test))

for fold, (trn_idx, val_idx) in enumerate(skf.split(X_full, y_full)):
    print(f"\n⚡ Fold {fold+1}/10")
    
    # 划分数据
    X_tr, X_val = X_full.iloc[trn_idx], X_full.iloc[val_idx]
    y_tr, y_val = y_full.iloc[trn_idx], y_full.iloc[val_idx]
    w_tr, w_val = weights[trn_idx], weights[val_idx]
    
    dtrain = xgb.DMatrix(X_tr, label=y_tr, weight=w_tr, enable_categorical=True)
    dvalid = xgb.DMatrix(X_val, label=y_val, weight=w_val, enable_categorical=True)
    dtest = xgb.DMatrix(X_test, enable_categorical=True)
    
    model = xgb.train(
        xgb_params,
        dtrain,
        num_boost_round=10000,
        evals=[(dvalid, 'valid')],
        early_stopping_rounds=200,
        verbose_eval=1000
    )
    
    oof_combined[val_idx] = model.predict(dvalid)
    pred_test += model.predict(dtest) / 10

oof_train = oof_combined[:n_train]
y_train_true = y_full[:n_train]

full_auc = roc_auc_score(y_train_true, oof_train)

# 计算 Cutoff AUC
cutoff_mask_indices = train_raw['id'] >= CUTOFF_ID
cutoff_auc = roc_auc_score(y_train_true[cutoff_mask_indices], oof_train[cutoff_mask_indices])

print(f"\n{'='*50}")
print(f"📊 {MODEL_NAME} Results")
print(f" Full OOF AUC: {full_auc:.5f}")
print(f"⭐ Cutoff AUC: {cutoff_auc:.5f}")
print(f"{'='*50}")

# 保存
test_ids = pd.read_csv("/kaggle/input/playground-series-s5e12/test.csv")['id']
save_predictions(oof_train, pred_test, test_ids, MODEL_NAME)

In [ ]:
%%writefile train_cat_inter.py
import numpy as np
import pandas as pd
from catboost import CatBoostClassifier, Pool
from utils import load_data, get_folds, save_predictions, RANDOM_STATE

print("🚀 Training CatBoost with Explicit Interactions (Masaya's Bigram Strategy)...")

X, y, test = load_data()

def create_interactions(df):
    if 'systolic_bp' in df.columns and 'diastolic_bp' in df.columns:
        df['pulse_pressure'] = df['systolic_bp'] - df['diastolic_bp']
    
    if 'age' in df.columns:
        df['age_bin'] = pd.qcut(df['age'], q=10, labels=False, duplicates='drop').astype(str)
    
    if 'bmi' in df.columns:
        df['bmi_bin'] = pd.qcut(df['bmi'], q=10, labels=False, duplicates='drop').astype(str)
        
    if 'glucose_fasting' in df.columns:
        df['glucose_bin'] = pd.qcut(df['glucose_fasting'], q=5, labels=False, duplicates='drop').astype(str)
    
    if 'age_bin' in df.columns and 'gender' in df.columns:
        df['gender_age_inter'] = df['gender'].astype(str) + '_' + df['age_bin']
        
    if 'ethnicity' in df.columns and 'bmi_bin' in df.columns:
        df['ethnicity_bmi_inter'] = df['ethnicity'].astype(str) + '_' + df['bmi_bin']
        
    if 'education_level' in df.columns and 'smoking_status' in df.columns:
        df['edu_smoke_inter'] = df['education_level'].astype(str) + '_' + df['smoking_status'].astype(str)

    return df

print("🔧 Generating Interaction Features...")
X = create_interactions(X)
test = create_interactions(test)

cat_features = [col for col in X.columns if X[col].dtype == 'object' or X[col].dtype.name == 'category']

for col in cat_features:
    X[col] = X[col].astype(str).fillna("MISSING")
    test[col] = test[col].astype(str).fillna("MISSING")

cat_features_indices = [X.columns.get_loc(c) for c in cat_features]

print(f"✅ Features created. Total: {X.shape[1]}")
print(f"📊 Cat Features used: {cat_features}")


params = {
    'loss_function': 'Logloss',
    'eval_metric': 'AUC',
    'learning_rate': 0.03,
    'iterations': 3000,
    'depth': 6,             
    'l2_leaf_reg': 5,       
    'random_strength': 1,
    'bagging_temperature': 1,
    'task_type': 'CPU',     
    'verbose': 500,
    'random_state': RANDOM_STATE,
    'early_stopping_rounds': 100,
    'allow_writing_files': False
}

oof = np.zeros(len(X))
pred = np.zeros(len(test))

for fold, (trn_idx, val_idx) in enumerate(get_folds(X, y)):
    X_tr, X_val = X.iloc[trn_idx], X.iloc[val_idx]
    y_tr, y_val = y.iloc[trn_idx], y.iloc[val_idx]
    
    train_pool = Pool(X_tr, y_tr, cat_features=cat_features_indices)
    val_pool = Pool(X_val, y_val, cat_features=cat_features_indices)
    test_pool = Pool(test, cat_features=cat_features_indices)
    
    model = CatBoostClassifier(**params)
    model.fit(train_pool, eval_set=val_pool, use_best_model=True)
    
    oof[val_idx] = model.predict_proba(val_pool)[:, 1]
    pred += model.predict_proba(test_pool)[:, 1] / 10
    
    print(f"Fold {fold+1}/10 Done")

save_predictions(oof, pred, test['id'], 'cat_inter')

In [ ]:
%%writefile train_cat_weighted.py
import numpy as np
import pandas as pd
from catboost import CatBoostClassifier, Pool
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
import warnings
from utils import save_predictions, TRAIN_PATH, ORIG_PATH, CUTOFF_ID, RANDOM_STATE

warnings.filterwarnings('ignore')

MODEL_NAME = 'cat_weighted'
print(f"🚀 Training CatBoost (Weighted Strategy: 1/16/8)...")

cat_params = {
    'iterations': 5000,
    'learning_rate': 0.02,
    'depth': 6,
    'loss_function': 'Logloss',
    'eval_metric': 'AUC',
    'random_seed': RANDOM_STATE,
    'l2_leaf_reg': 5,            # 增加正则化以应对强权重
    'allow_writing_files': False,
    'thread_count': -1,          # 使用所有CPU核心
    'task_type': 'CPU',          # TPU VM 只有 CPU 可用
    'verbose': 1000
}

def preprocessing_repro():
    train = pd.read_csv(TRAIN_PATH)
    test = pd.read_csv("/kaggle/input/playground-series-s5e12/test.csv")
    orig = pd.read_csv(ORIG_PATH).dropna()
    
    if 'Diabetes_binary' in orig.columns:
        orig = orig.rename(columns={'Diabetes_binary': 'diagnosed_diabetes'})
    
    common_cols = [c for c in train.columns if c in orig.columns]
    orig = orig[common_cols]
    target = 'diagnosed_diabetes'
    
    n_train, n_test, n_orig = len(train), len(test), len(orig)
    
    # 拼接
    train_full = pd.concat([train, orig], axis=0, ignore_index=True)
    full_df = pd.concat([train_full, test], axis=0, ignore_index=True)
    
    cat_features = full_df.select_dtypes(include=['object', 'bool']).columns.tolist()
    extra_cats = ['family_history_diabetes', 'hypertension_history', 'cardiovascular_history']
    for c in extra_cats:
        if c not in cat_features and c in full_df.columns:
            cat_features.append(c)
            
    num_features = [c for c in full_df.columns if c not in cat_features and c != target and c != 'id']
    
    print("🔧 Applying Feature Engineering (Orig Mean/Count)...")
    global_mean = orig[target].mean()
    
    for c in num_features + cat_features:
        mean_map = orig.groupby(c)[target].mean()
        full_df[f'{c}_org_mean'] = full_df[c].map(mean_map).fillna(global_mean)
        
        count_map = orig.groupby(c)[target].count()
        full_df[f'{c}_org_count'] = full_df[c].map(count_map).fillna(0)
        
    for c in cat_features:
        freqs = full_df[c].value_counts(normalize=True)
        full_df[f"{c}_fe"] = full_df[c].map(freqs)
        # CatBoost 更喜欢 fillna 后的 string
        full_df[c] = full_df[c].fillna("MISSING").astype(str)
        
    X_train_full = full_df.iloc[:n_train + n_orig].copy()
    X_test = full_df.iloc[n_train + n_orig:].copy()
    
    return X_train_full, X_test, n_train, n_orig, cat_features

X_full, X_test, n_train, n_orig, cat_cols = preprocessing_repro()
y_full = X_full['diagnosed_diabetes']
X_full = X_full.drop(columns=['diagnosed_diabetes', 'id'])
X_test = X_test.drop(columns=['diagnosed_diabetes', 'id'])

print("⚖️ Constructing Weights...")
train_raw = pd.read_csv(TRAIN_PATH)
tail_mask = train_raw['id'] >= CUTOFF_ID

w_train = np.ones(n_train)
w_train[tail_mask] = 16  # Tail 权重
w_orig = np.full(n_orig, 8) # Orig 权重
weights = np.concatenate([w_train, w_orig])

skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=RANDOM_STATE)

oof_combined = np.zeros(len(X_full))
pred_test = np.zeros(len(X_test))

# 提前准备 Test Pool
pool_test = Pool(X_test, cat_features=cat_cols)

for fold, (trn_idx, val_idx) in enumerate(skf.split(X_full, y_full)):
    print(f"\n⚡ Fold {fold+1}/10")
    
    X_tr, X_val = X_full.iloc[trn_idx], X_full.iloc[val_idx]
    y_tr, y_val = y_full.iloc[trn_idx], y_full.iloc[val_idx]
    w_tr, w_val = weights[trn_idx], weights[val_idx]
    
    train_pool = Pool(X_tr, y_tr, cat_features=cat_cols, weight=w_tr)
    valid_pool = Pool(X_val, y_val, cat_features=cat_cols, weight=w_val)
    
    model = CatBoostClassifier(**cat_params)
    
    model.fit(
        train_pool,
        eval_set=valid_pool,
        early_stopping_rounds=200,
        verbose=1000,
        use_best_model=True
    )
    
    oof_combined[val_idx] = model.predict_proba(valid_pool)[:, 1]
    pred_test += model.predict_proba(pool_test)[:, 1] / 10

oof_train = oof_combined[:n_train]
y_train_true = y_full[:n_train]

full_auc = roc_auc_score(y_train_true, oof_train)
cutoff_mask_indices = train_raw['id'] >= CUTOFF_ID
cutoff_auc = roc_auc_score(y_train_true[cutoff_mask_indices], oof_train[cutoff_mask_indices])

print(f"\n{'='*50}")
print(f"📊 {MODEL_NAME} Results")
print(f" Full OOF AUC: {full_auc:.5f}")
print(f"⭐ Cutoff AUC: {cutoff_auc:.5f}")
print(f"{'='*50}")

test_ids = pd.read_csv("/kaggle/input/playground-series-s5e12/test.csv")['id']
save_predictions(oof_train, pred_test, test_ids, MODEL_NAME)

In [ ]:
%%writefile train_tail_lr.py
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from scipy.stats import rankdata 
from utils import load_data, encode_categorical, save_predictions, TRAIN_PATH, RANDOM_STATE, CUTOFF_ID

print("🚀 Training Tail LR ...")

X, y, test = load_data()
X, X_test = encode_categorical(X.copy(), test.copy())

train_raw = pd.read_csv(TRAIN_PATH)
sorted_idx = train_raw['id'].argsort().values
TAIL_RATIO = 0.10
tail_start = int(len(sorted_idx) * (1 - TAIL_RATIO))
tail_idx = sorted_idx[tail_start:]

X_tail = X.iloc[tail_idx].reset_index(drop=True)
y_tail = y.iloc[tail_idx].reset_index(drop=True)
print(f"   Tail samples: {len(X_tail)} ({TAIL_RATIO*100:.0f}%)")

scaler = StandardScaler()
X_tail_scaled = scaler.fit_transform(X_tail)
X_scaled = scaler.transform(X)      
X_test_scaled = scaler.transform(X_test)

N_FOLDS = 100
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_STATE)
oof_tail_cv = np.zeros(len(X_tail))  
pred_test_avg = np.zeros(len(X_test))

print(f"   Starting {N_FOLDS}-fold CV to stabilize predictions...")

for fold, (trn_idx, val_idx) in enumerate(skf.split(X_tail_scaled, y_tail)):
    X_tr, X_val = X_tail_scaled[trn_idx], X_tail_scaled[val_idx]
    y_tr, y_val = y_tail.iloc[trn_idx], y_tail.iloc[val_idx]
    
    model = LogisticRegression(C=0.1, max_iter=1000, random_state=RANDOM_STATE)
    model.fit(X_tr, y_tr)
    
    oof_tail_cv[val_idx] = model.predict_proba(X_val)[:, 1]
    
    pred_test_avg += model.predict_proba(X_test_scaled)[:, 1] / N_FOLDS
    
    if (fold + 1) % 20 == 0:
        print(f"   Fold {fold+1}/{N_FOLDS} done.")

print(f"✅ Tail In-Sample AUC: {roc_auc_score(y_tail, oof_tail_cv):.5f}")

final_model = LogisticRegression(C=0.1, max_iter=1000, random_state=RANDOM_STATE)
final_model.fit(X_tail_scaled, y_tail)

oof_full_dirty = final_model.predict_proba(X_scaled)[:, 1]

print("🔧 Applying Quantile Mapping to fix Distribution & KS...")

def quantile_map(source_preds, target_dist):
    rank = rankdata(source_preds, method='ordinal') - 1
    pct = rank / (len(source_preds) - 1)
    
    target_sorted = np.sort(target_dist)
    
    mapped = np.interp(pct, np.linspace(0, 1, len(target_sorted)), target_sorted)
    return mapped

oof_full_mapped = quantile_map(oof_full_dirty, oof_tail_cv)
pred_test_mapped = quantile_map(pred_test_avg, oof_tail_cv)

oof_final = oof_full_mapped.copy()
oof_final[tail_idx] = oof_tail_cv

full_auc = roc_auc_score(y, oof_final)
cutoff_mask = train_raw['id'].values >= CUTOFF_ID
cutoff_auc = roc_auc_score(y[cutoff_mask], oof_final[cutoff_mask])

print(f"📊 Full OOF AUC: {full_auc:.5f}")
print(f"⭐ Cutoff AUC:   {cutoff_auc:.5f}")
print(f"   Mean Check: OOF Mean={oof_final.mean():.4f}, Target Mean={y.mean():.4f}")

save_predictions(oof_final, pred_test_mapped, test['id'], 'tail_lr')

In [ ]:
%%writefile train_tail_gam.py
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from pygam import LogisticGAM, s, f, l
from scipy.stats import rankdata
import warnings

warnings.filterwarnings('ignore')

from utils import load_data, save_predictions, TRAIN_PATH, TEST_PATH, RANDOM_STATE, CUTOFF_ID

MODEL_NAME = 'tail_gam'
print(f"🚀 Training Tail-Only GAM (Generalized Additive Model) ({MODEL_NAME})...")

X, y, test = load_data()

print("   Handling NaNs and Encoding...")
cat_cols = X.select_dtypes(include=['object', 'category']).columns.tolist()
num_cols = X.select_dtypes(include=['number']).columns.tolist()

for col in num_cols:
    med = X[col].median()
    X[col] = X[col].fillna(med)
    test[col] = test[col].fillna(med)

label_encoders = {}
for col in cat_cols:
    X[col] = X[col].fillna("MISSING").astype(str)
    test[col] = test[col].fillna("MISSING").astype(str)
    
    le = LabelEncoder()
    full_series = pd.concat([X[col], test[col]], axis=0).astype(str)
    le.fit(full_series)
    X[col] = le.transform(X[col])
    test[col] = le.transform(test[col])
    label_encoders[col] = le

train_raw = pd.read_csv(TRAIN_PATH)
tail_mask = train_raw['id'].values >= CUTOFF_ID

X_tail = X[tail_mask].reset_index(drop=True)
y_tail = y[tail_mask].reset_index(drop=True)

print(f"   Training on Tail data only. Samples: {len(X_tail)}")

scaler = StandardScaler()
X_tail[num_cols] = scaler.fit_transform(X_tail[num_cols])
X[num_cols] = scaler.transform(X[num_cols])
test[num_cols] = scaler.transform(test[num_cols])

feature_order = num_cols + cat_cols
X_tail = X_tail[feature_order]
X = X[feature_order]
test = test[feature_order]

term = None
print("   Building GAM terms...")
for i, col in enumerate(X_tail.columns):
    if col in num_cols:
        current_term = s(i, n_splines=20, lam=0.6) 
    elif col in cat_cols:
        current_term = f(i) 
    
    if term is None:
        term = current_term
    else:
        term += current_term

N_FOLDS = 10
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_STATE)
oof_tail = np.zeros(len(X_tail))
pred_test_avg = np.zeros(len(test))

print(f"   Starting {N_FOLDS}-fold CV on Tail...")

for fold, (trn_idx, val_idx) in enumerate(skf.split(X_tail, y_tail)):
    X_tr, X_val = X_tail.iloc[trn_idx].values, X_tail.iloc[val_idx].values
    y_tr, y_val = y_tail.iloc[trn_idx].values, y_tail.iloc[val_idx].values
    
    gam = LogisticGAM(term).fit(X_tr, y_tr)
    
    oof_tail[val_idx] = gam.predict_proba(X_val)
    pred_test_avg += gam.predict_proba(test.values) / N_FOLDS
    
    if (fold + 1) % 2 == 0:
        print(f"      Fold {fold+1} done.")

print(f"✅ Tail In-Sample CV AUC: {roc_auc_score(y_tail, oof_tail):.5f}")

print("   Fitting Final Model on Full Tail...")
final_gam = LogisticGAM(term).fit(X_tail.values, y_tail.values)

print("   Predicting Full Train (Dirty)...")
oof_full_dirty = final_gam.predict_proba(X.values)

print("🔧 Applying Quantile Mapping to fix Distribution & KS...")

def quantile_map(source_preds, target_dist):
    rank = rankdata(source_preds, method='ordinal') - 1
    pct = rank / (len(source_preds) - 1)
    target_sorted = np.sort(target_dist)
    mapped = np.interp(pct, np.linspace(0, 1, len(target_sorted)), target_sorted)
    return mapped

oof_final = quantile_map(oof_full_dirty, oof_tail)
pred_test_final = quantile_map(pred_test_avg, oof_tail)

oof_final[tail_mask] = oof_tail

full_auc = roc_auc_score(y, oof_final)
cutoff_auc = roc_auc_score(y[tail_mask], oof_final[tail_mask])
print(f"📊 Full OOF AUC (Fixed): {full_auc:.5f} (Previously ~0.52)")
print(f"⭐ Cutoff AUC:   {cutoff_auc:.5f}")

save_predictions(oof_final, pred_test_final, pd.read_csv(TEST_PATH)['id'], MODEL_NAME)

In [ ]:
%%writefile train_xgb_binned.py
import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import QuantileTransformer
from utils import load_data, save_predictions, TRAIN_PATH, ORIG_PATH, CUTOFF_ID, RANDOM_STATE

MODEL_NAME = 'xgb_binned'
print(f"🚀 Training {MODEL_NAME} (Multi-Binning + TE from Orig)...")

# =============================================================================
# 1. 数据加载 (不使用 load_data 的 orig 特征，自己处理)
# =============================================================================
train = pd.read_csv(TRAIN_PATH)
test = pd.read_csv("/kaggle/input/playground-series-s5e12/test.csv")
orig = pd.read_csv(ORIG_PATH).dropna()

if 'Diabetes_binary' in orig.columns:
    orig = orig.rename(columns={'Diabetes_binary': 'diagnosed_diabetes'})

y = train['diagnosed_diabetes']
target = 'diagnosed_diabetes'

# =============================================================================
# 2. 多种分箱方式 (来自 2nd place Loan Payback)
# =============================================================================
# 高基数数值列
high_card_nums = ['age', 'bmi', 'systolic_bp', 'diastolic_bp',
                  'cholesterol_total', 'ldl_cholesterol', 'hdl_cholesterol',
                  'triglycerides', 'waist_to_hip_ratio', 'heart_rate',
                  'physical_activity_minutes_per_week', 'alcohol_consumption_per_week']

n_train = len(train)
n_test = len(test)

full_df = pd.concat([train.drop(columns=[target]), test], axis=0, ignore_index=True)

print("🔧 Creating Multiple Binning Features...")

for col in high_card_nums:
    if col not in full_df.columns:
        continue
    
    # 1. 分位数分箱
    try:
        full_df[f'{col}_qbin'] = pd.qcut(full_df[col], q=50, labels=False, duplicates='drop').astype(str)
    except:
        pass
    
    # 2. 均匀分箱
    try:
        full_df[f'{col}_ubin'] = pd.cut(full_df[col], bins=50, labels=False).astype(str)
    except:
        pass
    
    # 3. Round + 整数除法 (模拟数字位)
    full_df[f'{col}_round_div5'] = (full_df[col].round() // 5).astype(int).astype(str)
    
    # 4. 直接转 int
    full_df[f'{col}_int'] = full_df[col].astype(int).astype(str)

# =============================================================================
# 3. 基于 Orig 的 Target Encoding
# =============================================================================
print("🔧 Target Encoding from Original Data...")
global_mean = orig[target].mean()

te_cols = [c for c in full_df.columns if c.endswith(('_qbin', '_ubin', '_round_div5', '_int'))]
cat_cols = full_df.select_dtypes(include=['object']).columns.tolist()
all_te_cols = list(set(te_cols + cat_cols))

for col in all_te_cols:
    if col in orig.columns or col.split('_')[0] in orig.columns:
        # 找到原始列名
        orig_col = col if col in orig.columns else col.split('_')[0]
        if orig_col in orig.columns:
            mean_map = orig.groupby(orig_col)[target].mean()
            full_df[f'{col}_te_orig'] = full_df[orig_col].map(mean_map).fillna(global_mean)

# =============================================================================
# 4. 处理类别特征
# =============================================================================
cat_cols_final = full_df.select_dtypes(include=['object']).columns.tolist()
for col in cat_cols_final:
    full_df[col] = full_df[col].astype('category')

# 分离 Train/Test
X = full_df.iloc[:n_train].copy()
X_test = full_df.iloc[n_train:].copy()

# 去掉 id
X = X.drop(columns=['id'], errors='ignore')
X_test = X_test.drop(columns=['id'], errors='ignore')

print(f"   Final Features: {X.shape[1]}")

# =============================================================================
# 5. 权重构造
# =============================================================================
train_raw = pd.read_csv(TRAIN_PATH)
train_ids = train_raw['id'].values
tail_mask = train_ids >= CUTOFF_ID

weights = np.ones(len(X))
weights[tail_mask] = 16.0

# =============================================================================
# 6. 训练
# =============================================================================
params = {
    'n_estimators': 10000,
    'learning_rate': 0.015,
    'max_depth': 4,
    'subsample': 0.6,
    'colsample_bytree': 0.4,
    'min_child_weight': 15,
    'reg_alpha': 8.0,
    'reg_lambda': 12.0,
    'eval_metric': 'auc',
    'tree_method': 'hist',
    'device': 'cpu',
    'random_state': RANDOM_STATE,
    'n_jobs': 4,
    'verbosity': 0,
    'enable_categorical': True
}

skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=RANDOM_STATE)
oof = np.zeros(len(X))
preds = np.zeros(len(X_test))

for fold, (trn_idx, val_idx) in enumerate(skf.split(X, y)):
    print(f"\n⚡ Fold {fold+1}/10")
    
    dtrain = xgb.DMatrix(X.iloc[trn_idx], label=y.iloc[trn_idx], 
                         weight=weights[trn_idx], enable_categorical=True)
    dvalid = xgb.DMatrix(X.iloc[val_idx], label=y.iloc[val_idx], enable_categorical=True)
    dtest = xgb.DMatrix(X_test, enable_categorical=True)
    
    model = xgb.train(
        params, dtrain, num_boost_round=10000,
        evals=[(dvalid, 'valid')],
        early_stopping_rounds=200, verbose_eval=1000
    )
    
    oof[val_idx] = model.predict(dvalid)
    preds += model.predict(dtest) / 10
    
    print(f"✅ Fold {fold+1} AUC: {roc_auc_score(y.iloc[val_idx], oof[val_idx]):.5f}")

# =============================================================================
# 7. 保存
# =============================================================================
full_auc = roc_auc_score(y, oof)
cutoff_auc = roc_auc_score(y[tail_mask], oof[tail_mask])

print(f"\n{'='*50}")
print(f"📊 {MODEL_NAME} Results")
print(f"   Full OOF AUC: {full_auc:.5f}")
print(f"⭐ Cutoff AUC:   {cutoff_auc:.5f}")
print(f"{'='*50}")

save_predictions(oof, preds, test['id'], MODEL_NAME)

In [ ]:
%%writefile train_xgb_pl.py
import os
import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import roc_auc_score
from utils import save_predictions, TRAIN_PATH, ORIG_PATH, CUTOFF_ID, RANDOM_STATE

MODEL_NAME = 'xgb_pl_soft'
BEST_SUB_FILE = "/kaggle/input/ps-s5e12-oofs-subs/submission_0_70774.csv" 

print(f"🚀 Training XGBoost with Soft Pseudo-Labeling...")

# 1. 加载数据
train = pd.read_csv(TRAIN_PATH)
test = pd.read_csv("/kaggle/input/playground-series-s5e12/test.csv")
orig = pd.read_csv(ORIG_PATH).dropna()
if 'Diabetes_binary' in orig.columns:
    orig = orig.rename(columns={'Diabetes_binary': 'diagnosed_diabetes'})

common_cols = [c for c in train.columns if c in orig.columns and c not in ['id', 'diagnosed_diabetes']]

y_train = train['diagnosed_diabetes']
y_orig = orig['diagnosed_diabetes']
train_ids = train['id'].values
test_ids = test['id'].values

# 2. 加载伪标签
if not os.path.exists(BEST_SUB_FILE):
    raise FileNotFoundError("Please run ensemble first!")

pseudo_df = pd.read_csv(BEST_SUB_FILE)
y_test_pl = pseudo_df['diagnosed_diabetes'].values
print(f"   Loaded Pseudo Labels. Mean: {y_test_pl.mean():.4f}")

# 3. 数据编码
full_df = pd.concat([train[common_cols], orig[common_cols], test[common_cols]], axis=0, ignore_index=True)
cat_cols = full_df.select_dtypes(include=['object', 'category']).columns

for col in cat_cols:
    le = LabelEncoder()
    full_df[col] = le.fit_transform(full_df[col].astype(str))

X_train = full_df.iloc[:len(train)].reset_index(drop=True)
X_orig = full_df.iloc[len(train):len(train)+len(orig)].reset_index(drop=True)
X_test_pl = full_df.iloc[len(train)+len(orig):].reset_index(drop=True)

# 4. 权重
tail_mask = train_ids >= CUTOFF_ID
w_train = np.ones(len(train))
w_train[tail_mask] = 16.0
w_orig = np.full(len(orig), 8.0)
w_test = np.full(len(test), 1.0)

# 5. 【关键修复】使用 DMatrix 和自定义训练
# 对于软标签，我们用 reg:squarederror 回归目标，然后 clip 输出

xgb_params = {
    'objective': 'reg:squarederror',  # 回归目标，可处理连续标签
    'eval_metric': 'rmse',
    'learning_rate': 0.01,
    'max_depth': 6,
    'subsample': 0.8,
    'colsample_bytree': 0.7,
    'tree_method': 'hist',
    'random_state': RANDOM_STATE,
    'n_jobs': 4,
    'verbosity': 0,
    'device': 'cpu'
}

skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=RANDOM_STATE)
oof_train = np.zeros(len(train))
pred_test_final = np.zeros(len(test))

# 拼接 Orig + Test (带软标签)
X_extra = pd.concat([X_orig, X_test_pl], axis=0).reset_index(drop=True)
y_extra = np.concatenate([y_orig.values.astype(float), y_test_pl.astype(float)])
w_extra = np.concatenate([w_orig, w_test])

for fold, (trn_idx, val_idx) in enumerate(skf.split(X_train, y_train)):
    print(f"\n⚡ Fold {fold+1}/10")
    
    X_tr_base = X_train.iloc[trn_idx].reset_index(drop=True)
    y_tr_base = y_train.iloc[trn_idx].values.astype(float)
    w_tr_base = w_train[trn_idx]
    
    X_val = X_train.iloc[val_idx].reset_index(drop=True)
    y_val = y_train.iloc[val_idx].values.astype(float)
    
    # 拼接
    X_tr_final = pd.concat([X_tr_base, X_extra], axis=0).reset_index(drop=True)
    y_tr_final = np.concatenate([y_tr_base, y_extra])
    w_tr_final = np.concatenate([w_tr_base, w_extra])
    
    dtrain = xgb.DMatrix(X_tr_final, label=y_tr_final, weight=w_tr_final)
    dvalid = xgb.DMatrix(X_val, label=y_val)
    dtest = xgb.DMatrix(X_test_pl)
    
    model = xgb.train(
        xgb_params,
        dtrain,
        num_boost_round=8000,
        evals=[(dvalid, 'valid')],
        early_stopping_rounds=200,
        verbose_eval=1000
    )
    
    # 预测并 clip 到 [0, 1]
    val_preds = model.predict(dvalid)
    val_preds = np.clip(val_preds, 0, 1)
    oof_train[val_idx] = val_preds
    
    test_preds = model.predict(dtest)
    test_preds = np.clip(test_preds, 0, 1)
    pred_test_final += test_preds / 10

full_auc = roc_auc_score(y_train, oof_train)
cutoff_auc = roc_auc_score(y_train[tail_mask], oof_train[tail_mask])

print(f"\n{'='*50}")
print(f"📊 {MODEL_NAME} Full OOF AUC: {full_auc:.5f} | Cutoff AUC: {cutoff_auc:.5f}")
save_predictions(oof_train, pred_test_final, test_ids, MODEL_NAME)

In [ ]:
%%writefile train_xgb_residual.py
import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import LabelEncoder
from utils import save_predictions, TRAIN_PATH, ORIG_PATH, CUTOFF_ID, RANDOM_STATE

MODEL_NAME = 'xgb_residual'
print(f"🚀 Training XGB Residual Boosting over Tail Mean...")

train = pd.read_csv(TRAIN_PATH)
test = pd.read_csv("/kaggle/input/playground-series-s5e12/test.csv")
orig = pd.read_csv(ORIG_PATH).dropna()

if 'Diabetes_binary' in orig.columns:
    orig = orig.rename(columns={'Diabetes_binary': 'diagnosed_diabetes'})

y = train['diagnosed_diabetes']
y_orig = orig['diagnosed_diabetes']
train_ids = train['id'].values
test_ids = test['id'].values

common_cols = [c for c in train.columns if c in orig.columns and c not in ['id', 'diagnosed_diabetes']]

tail_mask = train_ids >= CUTOFF_ID
TAIL_MEAN = y[tail_mask].mean()

def prob_to_logit(p, eps=1e-7):
    p = np.clip(p, eps, 1 - eps)
    return np.log(p / (1 - p))

BASELINE_LOGIT = prob_to_logit(TAIL_MEAN)
print(f"   Tail Mean (Baseline): {TAIL_MEAN:.4f}")
print(f"   Baseline Logit: {BASELINE_LOGIT:.4f}")

n_train, n_orig, n_test = len(train), len(orig), len(test)

full_df = pd.concat([
    train[common_cols], 
    orig[common_cols], 
    test[common_cols]
], axis=0, ignore_index=True)

cat_cols = full_df.select_dtypes(include=['object', 'category']).columns.tolist()
for col in cat_cols:
    le = LabelEncoder()
    full_df[col] = le.fit_transform(full_df[col].astype(str))

# Orig TE
global_mean = orig['diagnosed_diabetes'].mean()
for col in common_cols:
    mean_map = orig.groupby(col)['diagnosed_diabetes'].mean()
    full_df[f'{col}_orig_te'] = full_df[col].map(mean_map).fillna(global_mean)

X = full_df.iloc[:n_train]
X_orig = full_df.iloc[n_train:n_train+n_orig]
X_test = full_df.iloc[n_train+n_orig:]

w_train = np.ones(n_train)
w_train[tail_mask] = 16.0
w_orig = np.full(n_orig, 8.0)

params = {
    'objective': 'binary:logistic',  # 仍然是分类
    'eval_metric': 'auc',
    'learning_rate': 0.01,
    'max_depth': 5,
    'subsample': 0.7,
    'colsample_bytree': 0.5,
    'min_child_weight': 10,
    'reg_alpha': 5.0,
    'reg_lambda': 10.0,
    'tree_method': 'hist',
    'device': 'cpu',
    'random_state': RANDOM_STATE,
    'n_jobs': 4,
    'verbosity': 0
}

skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=RANDOM_STATE)
oof = np.zeros(n_train)
preds = np.zeros(n_test)

for fold, (trn_idx, val_idx) in enumerate(skf.split(X, y)):
    print(f"\n⚡ Fold {fold+1}/10")
    
    # 拼接数据
    X_tr = pd.concat([X.iloc[trn_idx], X_orig], axis=0)
    y_tr = np.concatenate([y.values[trn_idx], y_orig.values])
    w_tr = np.concatenate([w_train[trn_idx], w_orig])
    
    X_val = X.iloc[val_idx]
    
    base_margin_tr = np.full(len(X_tr), BASELINE_LOGIT)
    base_margin_val = np.full(len(X_val), BASELINE_LOGIT)
    base_margin_test = np.full(n_test, BASELINE_LOGIT)
    
    dtrain = xgb.DMatrix(X_tr, label=y_tr, weight=w_tr, base_margin=base_margin_tr)
    dvalid = xgb.DMatrix(X_val, label=y.values[val_idx], base_margin=base_margin_val)
    dtest = xgb.DMatrix(X_test, base_margin=base_margin_test)
    
    model = xgb.train(
        params, dtrain, num_boost_round=10000,
        evals=[(dvalid, 'valid')],
        early_stopping_rounds=200, verbose_eval=1000
    )
    
    oof[val_idx] = model.predict(dvalid)
    preds += model.predict(dtest) / 10

full_auc = roc_auc_score(y, oof)
cutoff_auc = roc_auc_score(y[tail_mask], oof[tail_mask])

print(f"\n{'='*50}")
print(f"📊 {MODEL_NAME} Results")
print(f"   Full OOF AUC: {full_auc:.5f}")
print(f"⭐ Cutoff AUC:   {cutoff_auc:.5f}")
print(f"   Pred Mean: {preds.mean():.4f} (Baseline: {TAIL_MEAN:.4f})")
print(f"{'='*50}")

save_predictions(oof, preds, test_ids, MODEL_NAME)

# ENS

In [ ]:
import numpy as np
import pandas as pd
import os
from scipy.stats import ks_2samp
from sklearn.linear_model import Ridge
from sklearn.metrics import roc_auc_score
from sklearn.utils import resample
import warnings
warnings.filterwarnings('ignore')

TRAIN_PATH = "/kaggle/input/playground-series-s5e12/train.csv"
TEST_PATH = "/kaggle/input/playground-series-s5e12/test.csv"
OOF_DIR = "/kaggle/input/ps-s5e12-oofs-subs" 
OUTPUT_DIR = "outputs"
CUTOFF_ID = 678260

# 模型列表
MODELS = ['lgb', 
          'lgb_te', 
          'lgb_safe',
          'lgb_pseudo',
          
          'cat_weighted',
          'cat_inter', 
          
          'tail_lr',
          'tail_gam',
          
          'nn_dae',
          'nn_stacking',
          
          'xgb',
          'xgb_binned',
          'xgb_pl_soft',
          'xgb_residual',
         ]

print("="*70)
print("📊 ENSEMBLE: ")
print("="*70)

train = pd.read_csv(TRAIN_PATH)
test = pd.read_csv(TEST_PATH)
y_true = train['diagnosed_diabetes'].values
train_ids = train['id'].values
test_ids = test['id'].values

cutoff_mask = train_ids >= CUTOFF_ID
n_cutoff = cutoff_mask.sum()

print(f"\n📌 Cutoff ID: {CUTOFF_ID}")
print(f"📌 Cutoff samples: {n_cutoff} ({n_cutoff/len(train)*100:.1f}%)")

oofs = {}
subs = {}

print(f"\n{'Model':<15} {'Full AUC':>10} {'Cutoff AUC':>12} {'KS Stat':>10} {'Status'}")
print("-"*60)

for model in MODELS:
    oof_path = f"{OUTPUT_DIR}/oof_{model}.csv"
    sub_path = f"{OUTPUT_DIR}/submission_{model}.csv"

    if not os.path.exists(oof_path):
        oof_path = f"{OOF_DIR}/oof_{model}.csv"
        sub_path = f"{OOF_DIR}/submission_{model}.csv"
    
    if os.path.exists(oof_path) and os.path.exists(sub_path):
        oof = pd.read_csv(oof_path)['oof'].values
        sub = pd.read_csv(sub_path)['diagnosed_diabetes'].values
        
        oofs[model] = oof
        subs[model] = sub
        
        full_auc = roc_auc_score(y_true, oof)
        cutoff_auc = roc_auc_score(y_true[cutoff_mask], oof[cutoff_mask])
        ks_stat, _ = ks_2samp(oof, sub)
        
        print(f"{model:<15} {full_auc:>10.5f} {cutoff_auc:>12.5f} {ks_stat:>10.4f} ✅")
    else:
        print(f"{model:<15} {'--':>10} {'--':>12} {'--':>10} ❌ Not found")

print("-"*60)
print(f"✅ Loaded {len(oofs)} models\n")

if len(oofs) == 0:
    raise ValueError("No OOF files found!")

model_names = list(oofs.keys())
X_oof = np.column_stack([oofs[m] for m in model_names])
X_sub = np.column_stack([subs[m] for m in model_names])

X_oof_cutoff = X_oof[cutoff_mask]
y_cutoff = y_true[cutoff_mask]

print("="*70)
print("🔧 Method 1: Ridge Regression ")
print("="*70)

best_alpha = None
best_cutoff_auc = 0

for alpha in [0.001, 0.01, 0.1, 1, 10, 100]:
    ridge = Ridge(alpha=alpha, fit_intercept=False, positive=True)
    ridge.fit(X_oof_cutoff, y_cutoff)

    w = ridge.coef_
    if w.sum() > 0: w = w / w.sum()
    
    cutoff_pred = X_oof_cutoff @ w
    cutoff_auc = roc_auc_score(y_cutoff, cutoff_pred)
    
    if cutoff_auc > best_cutoff_auc:
        best_cutoff_auc = cutoff_auc
        best_alpha = alpha

print(f"   Best alpha: {best_alpha}")

ridge = Ridge(alpha=best_alpha, fit_intercept=False, positive=True)
ridge.fit(X_oof_cutoff, y_cutoff)

w_ridge = ridge.coef_
if w_ridge.sum() > 0: w_ridge = w_ridge / w_ridge.sum()

oof_ridge = X_oof @ w_ridge
sub_ridge = X_sub @ w_ridge

full_auc_ridge = roc_auc_score(y_true, oof_ridge)
cutoff_auc_ridge = roc_auc_score(y_cutoff, oof_ridge[cutoff_mask])
ks_ridge, _ = ks_2samp(oof_ridge, sub_ridge)

print(f"\n   Ridge (Pos) Weights:")
for i, name in enumerate(model_names):
    print(f"      {name:<15}: {w_ridge[i]:>8.4f}")

print(f"\n   📊 Ridge Results:")
print(f"      Full OOF AUC:   {full_auc_ridge:.5f}")
print(f"      Cutoff AUC:     {cutoff_auc_ridge:.5f}")
print(f"      KS Statistic:   {ks_ridge:.4f}")

print("\n" + "="*70)
print("🔧 Method 2: Bagged Hill Climbing ")
print("="*70)

def bagged_hill_climbing(X, y, n_bags=50, n_iter=2000, step=0.01, seed=42):
    np.random.seed(seed)
    n_models = X.shape[1]
    bag_weights = []
    
    print(f"   Running Bagged HC with {n_bags} bags...")
    
    for i in range(n_bags):
        X_s, y_s = resample(X, y, replace=True, random_state=seed+i)
        w = np.ones(n_models) / n_models
        best = roc_auc_score(y_s, X_s @ w)
        
        for _ in range(n_iter):
            new_w = w.copy()
            idx = np.random.randint(n_models)
            delta = np.random.choice([-step, step])
            new_w[idx] += delta
            new_w = np.maximum(new_w, 0)
            if new_w.sum() == 0: continue
            new_w /= new_w.sum()
            
            sc = roc_auc_score(y_s, X_s @ new_w)
            if sc > best:
                best = sc
                w = new_w.copy()
        bag_weights.append(w)
        
        if (i+1) % 10 == 0:
            print(f"      Bag {i+1}/{n_bags} done")

    final_w = np.mean(bag_weights, axis=0)
    return final_w / final_w.sum()

weights_hc = bagged_hill_climbing(X_oof_cutoff, y_cutoff, n_bags=50, n_iter=2000, step=0.02)

oof_hc = X_oof @ weights_hc
sub_hc = X_sub @ weights_hc

full_auc_hc = roc_auc_score(y_true, oof_hc)
cutoff_auc_hc = roc_auc_score(y_cutoff, oof_hc[cutoff_mask])
ks_hc, _ = ks_2samp(oof_hc, sub_hc)

print(f"\n   Bagged HC Weights:")
for i, name in enumerate(model_names):
    print(f"      {name:<15}: {weights_hc[i]:>8.4f}")

print(f"\n   📊 Bagged Hill Climbing Results:")
print(f"      Full OOF AUC:   {full_auc_hc:.5f}")
print(f"      Cutoff AUC:     {cutoff_auc_hc:.5f}")
print(f"      KS Statistic:   {ks_hc:.4f}")

print("\n" + "="*70)
print("🔧 Method 3: Simple Average (baseline)")
print("="*70)

oof_avg = X_oof.mean(axis=1)
sub_avg = X_sub.mean(axis=1)

full_auc_avg = roc_auc_score(y_true, oof_avg)
cutoff_auc_avg = roc_auc_score(y_cutoff, oof_avg[cutoff_mask])
ks_avg, _ = ks_2samp(oof_avg, sub_avg)

print(f"\n   📊 Simple Average Results:")
print(f"      Full OOF AUC:   {full_auc_avg:.5f}")
print(f"      Cutoff AUC:     {cutoff_auc_avg:.5f}")
print(f"      KS Statistic:   {ks_avg:.4f}")

# --- Summary & Save ---
print("\n" + "="*70)
print("📊 SUMMARY COMPARISON")
print("="*70)
print(f"\n{'Method':<20} {'Full AUC':>12} {'Cutoff AUC':>12} {'KS Stat':>10}")
print("-"*56)
print(f"{'Simple Average':<20} {full_auc_avg:>12.5f} {cutoff_auc_avg:>12.5f} {ks_avg:>10.4f}")
print(f"{'Ridge (Pos)':<20} {full_auc_ridge:>12.5f} {cutoff_auc_ridge:>12.5f} {ks_ridge:>10.4f}")
print(f"{'Bagged HC':<20} {full_auc_hc:>12.5f} {cutoff_auc_hc:>12.5f} {ks_hc:>10.4f}")
print("-"*56)

results = {
    'avg': (cutoff_auc_avg, ks_avg, oof_avg, sub_avg),
    'ridge': (cutoff_auc_ridge, ks_ridge, oof_ridge, sub_ridge),
    'hc': (cutoff_auc_hc, ks_hc, oof_hc, sub_hc)
}

# 优先比较 AUC，如果 AUC 相同或极度接近，则比较 -KS 
best_method = max(results.keys(), key=lambda x: (results[x][0], -results[x][1]))
best_cutoff, best_ks, best_oof, best_sub = results[best_method]

print(f"\n🏆 Best Method: {best_method.upper()} (Cutoff AUC: {best_cutoff:.5f})")

os.makedirs(OUTPUT_DIR, exist_ok=True)

pd.DataFrame({'id': test_ids, 'diagnosed_diabetes': sub_ridge}).to_csv(
    f"{OUTPUT_DIR}/submission_ridge.csv", index=False)
pd.DataFrame({'id': test_ids, 'diagnosed_diabetes': sub_hc}).to_csv(
    f"{OUTPUT_DIR}/submission_hc.csv", index=False)
pd.DataFrame({'id': test_ids, 'diagnosed_diabetes': sub_avg}).to_csv(
    f"{OUTPUT_DIR}/submission_avg.csv", index=False)

pd.DataFrame({'id': test_ids, 'diagnosed_diabetes': best_sub}).to_csv(
    "submission.csv", index=False)

print(f"\n✅ Saved all ensemble results. Main submission: submission.csv (from {best_method})")